# 💃 Dancing Numbers — RAG Pipeline
Run cells **top to bottom, one by one**. Do NOT skip any cell.

In [1]:
# ═══════════════════════════════════════════════════════════
# CELL 0 — SET WORKING DIRECTORY (run this first, always)
# ═══════════════════════════════════════════════════════════
import os

# ── Paste your actual AI_agent folder path here ──
FOLDER = r"C:\Users\Kartik\Desktop\AI agent Chat"  # ← EDIT THIS

os.chdir(FOLDER)
print('✅ Working directory:', os.getcwd())

# Check file status
for f in ['blogs.db', 'blogs_faiss.index', 'chunks_map.npy']:
    status = '✅ exists' if os.path.exists(f) else '❌ missing'
    print(f'   {f:30s} {status}')

✅ Working directory: C:\Users\Kartik\Desktop\AI agent Chat
   blogs.db                       ✅ exists
   blogs_faiss.index              ❌ missing
   chunks_map.npy                 ❌ missing


In [2]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — INSTALL LIBRARIES
# ═══════════════════════════════════════════════════════════
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'requests', 'beautifulsoup4', 'sentence-transformers',
    'faiss-cpu', 'langchain', 'langchain-text-splitters',
    'tqdm', 'lxml', 'streamlit', '--quiet'
])
print('✅ Libraries installed')

✅ Libraries installed


In [3]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — IMPORTS
# ═══════════════════════════════════════════════════════════
import requests, sqlite3, time, re, warnings
import numpy as np
from bs4 import BeautifulSoup
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
except ImportError:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
warnings.filterwarnings('ignore')
print('✅ All imports successful')

✅ All imports successful


# ═══════════════════════════════════════════════════════════
# CELL 3 — CONFIGURATION
# ═══════════════════════════════════════════════════════════
SITEMAP_INDEX_URL = 'https://www.dancingnumbers.com/sitemap_index.xml'
DB_PATH           = 'blogs.db'
FAISS_INDEX_PATH  = 'blogs_faiss.index'
CHUNK_MAP_PATH    = 'chunks_map.npy'
EMBEDDING_MODEL   = 'all-MiniLM-L6-v2'
CHUNK_SIZE        = 500
CHUNK_OVERLAP     = 100
BATCH_SIZE        = 64
MAX_SITEMAPS      = 5
MAX_BLOG_URLS     = 2000
CRAWL_DELAY       = 0.5
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}
print('✅ Configuration set')
print(f'   Saving all files to: {os.getcwd()}')

In [5]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — FETCH SITEMAP INDEX
# ═══════════════════════════════════════════════════════════
def fetch_url(url, timeout=15):
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f'  ⚠️ Failed: {url} → {e}')
        return None

sitemap_xml = fetch_url(SITEMAP_INDEX_URL)
if sitemap_xml:
    print(f'✅ Sitemap fetched ({len(sitemap_xml):,} chars)')
else:
    raise RuntimeError('Cannot fetch sitemap — check internet connection')

✅ Sitemap fetched (1,874 chars)


In [6]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — EXTRACT CHILD SITEMAP URLs
# ═══════════════════════════════════════════════════════════
def extract_locs(xml_text):
    soup = BeautifulSoup(xml_text, 'lxml-xml')
    return [loc.get_text(strip=True) for loc in soup.find_all('loc')]

sitemap_urls = extract_locs(sitemap_xml)
print(f'✅ Found {len(sitemap_urls)} child sitemaps')
for u in sitemap_urls[:5]:
    print(f'   • {u}')

✅ Found 12 child sitemaps
   • https://www.dancingnumbers.com/post-sitemap1.xml
   • https://www.dancingnumbers.com/post-sitemap2.xml
   • https://www.dancingnumbers.com/post-sitemap3.xml
   • https://www.dancingnumbers.com/post-sitemap4.xml
   • https://www.dancingnumbers.com/post-sitemap5.xml


In [7]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — COLLECT BLOG URLs
# ═══════════════════════════════════════════════════════════
all_blog_urls = []

for sm_url in tqdm(sitemap_urls[:MAX_SITEMAPS], desc='Sitemaps'):
    xml = fetch_url(sm_url)
    if not xml:
        continue
    locs = extract_locs(xml)
    blog_urls = [
        u for u in locs
        if u.startswith('https://www.dancingnumbers.com/')
        and not any(x in u for x in ['/tag/','/category/','?p=','/page/','/author/'])
        and u != 'https://www.dancingnumbers.com/'
    ]
    all_blog_urls.extend(blog_urls)
    time.sleep(CRAWL_DELAY)

all_blog_urls = list(dict.fromkeys(all_blog_urls))[:MAX_BLOG_URLS]
print(f'\n✅ Collected {len(all_blog_urls)} blog URLs')

Sitemaps: 100%|██████████| 5/5 [00:09<00:00,  1.82s/it]


✅ Collected 852 blog URLs


In [8]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — CREATE DATABASE
# ═══════════════════════════════════════════════════════════
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
cur.execute('''
    CREATE TABLE IF NOT EXISTS blogs (
        id      INTEGER PRIMARY KEY AUTOINCREMENT,
        url     TEXT UNIQUE NOT NULL,
        title   TEXT,
        content TEXT
    )
''')
cur.execute('''
    CREATE TABLE IF NOT EXISTS chunks (
        chunk_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        blog_id    INTEGER NOT NULL,
        chunk_text TEXT NOT NULL,
        FOREIGN KEY (blog_id) REFERENCES blogs(id)
    )
''')
conn.commit()
conn.close()
print(f'✅ Database created: {os.path.abspath(DB_PATH)}')

✅ Database created: C:\Users\Kartik\Desktop\AI agent Chat\blogs.db


In [9]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — INSERT URLs INTO DATABASE
# ═══════════════════════════════════════════════════════════
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
inserted = 0
for url in all_blog_urls:
    cur.execute('INSERT OR IGNORE INTO blogs (url) VALUES (?)', (url,))
    if cur.rowcount > 0:
        inserted += 1
conn.commit()
conn.close()
print(f'✅ Inserted {inserted} URLs into blogs table')

✅ Inserted 802 URLs into blogs table


In [10]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — SCRAPE & STORE BLOG CONTENT
# ═══════════════════════════════════════════════════════════
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x20-\x7E\n]', '', text)
    return text.strip()

def scrape_blog(url):
    html = fetch_url(url)
    if not html:
        return None, None
    soup  = BeautifulSoup(html, 'html.parser')
    title = None
    h1    = soup.find('h1')
    if h1:
        title = h1.get_text(strip=True)
    elif soup.title:
        title = soup.title.get_text(strip=True)
    content_tag = None
    for sel in ['article','div.entry-content','div.post-content','div.content','main']:
        content_tag = soup.select_one(sel)
        if content_tag:
            break
    if content_tag:
        for n in content_tag.find_all(['nav','footer','aside','script','style']):
            n.decompose()
        content = content_tag.get_text(separator=' ', strip=True)
    else:
        content = ' '.join(p.get_text(strip=True) for p in soup.find_all('p'))
    content = clean_text(content)
    return (title, content) if len(content) >= 100 else (title, None)

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
cur.execute('SELECT id, url FROM blogs WHERE content IS NULL')
rows = cur.fetchall()
print(f'📄 Scraping {len(rows)} pages...')
ok = fail = 0
for blog_id, url in tqdm(rows, desc='Scraping'):
    title, content = scrape_blog(url)
    if content:
        cur.execute('UPDATE blogs SET title=?, content=? WHERE id=?', (title, content, blog_id))
        ok += 1
    else:
        fail += 1
    conn.commit()
    time.sleep(CRAWL_DELAY)
conn.close()
print(f'\n✅ Scraped {ok} pages ({fail} failed)')

📄 Scraping 828 pages...


Scraping: 100%|██████████| 828/828 [31:49<00:00,  2.31s/it]


✅ Scraped 827 pages (1 failed)


In [11]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — CHUNK CONTENT
# ═══════════════════════════════════════════════════════════
try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
except ImportError:
    from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n','\n','. ',' ','']
)
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
cur.execute('''
    SELECT b.id, b.content FROM blogs b
    LEFT JOIN chunks c ON b.id = c.blog_id
    WHERE b.content IS NOT NULL
    GROUP BY b.id HAVING COUNT(c.chunk_id) = 0
''')
rows = cur.fetchall()
total = 0
for blog_id, content in tqdm(rows, desc='Chunking'):
    for chunk in splitter.split_text(content):
        chunk = chunk.strip()
        if len(chunk) > 50:
            cur.execute('INSERT INTO chunks (blog_id, chunk_text) VALUES (?,?)', (blog_id, chunk))
            total += 1
conn.commit()
conn.close()
print(f'✅ Created {total:,} chunks')

Chunking: 100%|██████████| 851/851 [00:00<00:00, 3426.18it/s]

✅ Created 23,440 chunks


In [12]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — GENERATE EMBEDDINGS
# ═══════════════════════════════════════════════════════════
print(f'Loading model: {EMBEDDING_MODEL}')
model = SentenceTransformer(EMBEDDING_MODEL)

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
cur.execute('SELECT chunk_id, chunk_text FROM chunks ORDER BY chunk_id')
rows = cur.fetchall()
conn.close()

chunk_ids   = [r[0] for r in rows]
chunk_texts = [r[1] for r in rows]
print(f'Embedding {len(chunk_texts):,} chunks...')

embeddings = model.encode(
    chunk_texts, batch_size=BATCH_SIZE,
    show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True
)
print(f'✅ Embeddings shape: {embeddings.shape}')

Loading model: all-MiniLM-L6-v2
Embedding 23,440 chunks...


Batches:   0%|          | 0/367 [00:00<?, ?it/s]

✅ Embeddings shape: (23440, 384)


In [13]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — BUILD & SAVE FAISS INDEX
# ═══════════════════════════════════════════════════════════
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))

# Save both files
faiss.write_index(index, FAISS_INDEX_PATH)
np.save(CHUNK_MAP_PATH, np.array(chunk_ids))

# Verify files exist on disk
print('✅ FAISS index built and saved')
for f in [FAISS_INDEX_PATH, CHUNK_MAP_PATH, DB_PATH]:
    path = os.path.abspath(f)
    size = os.path.getsize(path) / 1024
    print(f'   ✅ {f:30s}  {size:,.0f} KB  →  {path}')

✅ FAISS index built and saved
   ✅ blogs_faiss.index               35,160 KB  →  C:\Users\Kartik\Desktop\AI agent Chat\blogs_faiss.index
   ✅ chunks_map.npy                  183 KB  →  C:\Users\Kartik\Desktop\AI agent Chat\chunks_map.npy
   ✅ blogs.db                        20,416 KB  →  C:\Users\Kartik\Desktop\AI agent Chat\blogs.db


In [14]:
# ═══════════════════════════════════════════════════════════
# CELL 13 — QUICK SEARCH TEST (verify everything works)
# ═══════════════════════════════════════════════════════════
# Reload from disk to confirm save/load cycle works
test_index    = faiss.read_index(FAISS_INDEX_PATH)
test_id_map   = np.load(CHUNK_MAP_PATH)

q_emb = model.encode(['how to fix payment issue quickbooks'],
                     convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
scores, indices = test_index.search(q_emb, 3)

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

print('🔍 Test search results:\n')
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    chunk_id = int(test_id_map[idx])
    cur.execute('''
        SELECT c.chunk_text, b.title, b.url
        FROM chunks c JOIN blogs b ON c.blog_id = b.id
        WHERE c.chunk_id = ?
    ''', (chunk_id,))
    row = cur.fetchone()
    if row:
        print(f'  Rank {rank} | Score: {score:.4f}')
        print(f'  Title: {row[1]}')
        print(f'  Chunk: {row[0][:150]}...')
        print()

conn.close()
print('✅ Pipeline complete! All 3 files are ready.')
print('\nNow run:  streamlit run app.py')

🔍 Test search results:

  Rank 1 | Score: 0.8384
  Title: How to Troubleshoot QuickBooks Payments (Merchant Services) Issues
  Chunk: . In this guide, common QuickBooks Payments Issues related to the Merchant Service have been shared. Readers need to go through each step and implemen...

  Rank 2 | Score: 0.8164
  Title: How to Troubleshoot QuickBooks Payments (Merchant Services) Issues
  Chunk: . 3. Repair Data Damage Review the company file corruption using QuickBooks tools. Fix any damage identified. Try to record deposits once more. 4. Rem...

  Rank 3 | Score: 0.8121
  Title: How to Import Invoices into QuickBooks from Excel?
  Chunk: . Learn the exact process in how to import payment items into QuickBooks Desktop . Locate the error location Update the fields with relevant detail to...

✅ Pipeline complete! All 3 files are ready.

Now run:  streamlit run app.py
